In [1]:
import pandas as pd
import boto3
import math
from sagemaker import get_execution_role
from pprint import pprint
import time

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Functions

In [2]:
def get_specs(str_instance):
    if str_instance == 'm5.large':
        int_vcpu = 2
        int_memory_gb = 8
    elif str_instance == 'm5.xlarge':
        int_vcpu = 4
        int_memory_gb = 16
    elif str_instance == 'm5.2xlarge':
        int_vcpu = 8
        int_memory_gb = 32
    elif str_instance == 'm5.4xlarge':
        int_vcpu = 16
        int_memory_gb = 64
    elif str_instance == 'm5.8xlarge':
        int_vcpu = 32
        int_memory_gb = 128
    elif str_instance == 'm5.12xlarge':
        int_vcpu = 48
        int_memory_gb = 192
    int_memory_mebibytes = math.ceil(int_memory_gb * 953.674)
    dict_output = {
        'int_vcpu': int_vcpu,
        'int_memory_gb': int_memory_gb,
        'int_memory_mebibytes': int_memory_mebibytes,
    }
    return dict_output

In [3]:
def get_list_of_files_in_s3_location(cls_client, str_bucket_name, str_prefix):
    list_str_filename = []
    continuation_token = None

    while True:
        if continuation_token:
            dict_response = cls_client.list_objects_v2(
                Bucket=str_bucket_name,
                Prefix=str_prefix,
                ContinuationToken=continuation_token
            )
        else:
            dict_response = cls_client.list_objects_v2(
                Bucket=str_bucket_name,
                Prefix=str_prefix
            )

        # Add the filenames from this batch
        list_dict_contents = dict_response.get('Contents', [])
        for dict_contents in list_dict_contents:
            file_key = dict_contents['Key']
            if 'gzip' in file_key:
                filename = file_key.split('/')[-1]
                list_str_filename.append(filename)

        # Check if there are more files to fetch
        continuation_token = dict_response.get('NextContinuationToken')
        if not continuation_token:
            break

    return list_str_filename

### Constants

In [4]:
str_image_name = 'simple-model-test-parse-snowflake'
int_iteration = 1
str_instance = 'm5.2xlarge'
dict_specs = get_specs(str_instance=str_instance)
int_vcpu = dict_specs['int_vcpu']
int_memory_gb = dict_specs['int_memory_gb']
int_memory_mebibytes = dict_specs['int_memory_mebibytes']
for key, val in dict_specs.items():
    print(f'{key}: {val}')

int_vcpu: 8
int_memory_gb: 32
int_memory_mebibytes: 30518


### Get number of jobs

In [5]:
cls_client = boto3.client('s3')
list_str_filename = get_list_of_files_in_s3_location(
    cls_client=cls_client,
    str_bucket_name='20241022-parse-snowflake-payloads',
    str_prefix='01_pull_payloads',
)
int_n_jobs = len(list_str_filename)
print(f'Number of Files: {int_n_jobs}')

Number of Files: 1135


### Save to s3

In [6]:
df = pd.DataFrame({'str_filename': list_str_filename})
str_filename = 'df_list_str_filename_2.csv'
str_uri = f's3://20241112-simple-model-test/filenames_for_parsing/{str_filename}'
df.to_csv(str_uri, index=False)
# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:279: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,str_filename
0,df_requests_2021-10-05.gzip
1,df_requests_2021-10-06.gzip
2,df_requests_2021-10-07.gzip
3,df_requests_2021-10-08.gzip
4,df_requests_2021-10-09.gzip
...,...
1130,df_requests_2024-11-09.gzip
1131,df_requests_2024-11-10.gzip
1132,df_requests_2024-11-11.gzip
1133,df_requests_2024-11-12.gzip


### Create compute environment

In [7]:
# initialize class
cls_client = boto3.client('batch')

In [8]:
# get role
try:
    str_role = get_execution_role()
except:
    ! pip install --upgrade boto3
    str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# create compute environment
while True:
    try:
        str_compute_env_name = f'env-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_compute_environment(
            computeEnvironmentName=str_compute_env_name,
            type= 'Managed', 
            state= 'ENABLED',
            serviceRole = str_role,
            computeResources={
                #'type': 'SPOT',
                'type': 'EC2',
                'minvCpus': 0,
                'maxvCpus': 256, 
                'desiredvCpus': int_vcpu,
                'instanceTypes': [
                    str_instance,
                ], 
                'subnets': ['subnet-044e573651bb251a7'], 
                'securityGroupIds': ['sg-03904237048cdc335'], 
                'instanceRole': 'ecsInstanceRole',
                #'spotIamFleetRole': 'AmazonEC2SpotFleetTaggingRole',
            },
        )
        pprint(dict_response)
        break
    except:
        int_iteration += 1

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '191',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 14 Nov 2024 19:15:44 GMT',
                                      'x-amz-apigw-id': 'BQDfFH8lvHcEB_g=',
                                      'x-amzn-requestid': '2300d7aa-acd8-4a15-a7af-c60b3d50bc3b',
                                      'x-amzn-trace-id': 'Root=1-67364c60-259af03567428f7e511e9d9a'},
                      'HTTPStatusCode': 200,
                      'RequestId': '2300d7aa-acd8-4a15-a7af-c60b3d50bc3b',
                      'RetryAttempts': 0},
 'computeEnvironmentArn': 'arn:aws:batch:

### Create Job Queue

In [10]:
# create job queue (this is where AWS will store your jobs until an EC2 Instance is available to run them)
while True:
    try:
        str_job_queue_name = f'queue-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_job_queue(
            jobQueueName=str_job_queue_name,
            state='ENABLED',
            priority=1,
            computeEnvironmentOrder=[
                {
                    'order': 1,
                    'computeEnvironment': str_compute_env_name,
                },
            ]
        )
        # get arn
        str_job_queue_arn = dict_response['jobQueueArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '165',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 14 Nov 2024 19:16:08 GMT',
                                      'x-amz-apigw-id': 'BQDi3GNwPHcEtSw=',
                                      'x-amzn-requestid': 'e085979a-e168-47e5-aefd-e62592228f82',
                                      'x-amzn-trace-id': 'Root=1-67364c78-1acae0e97bc6eabf6158d52b'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'e085979a-e168-47e5-aefd-e62592228f82',
                      'RetryAttempts': 0},
 'jobQueueArn': 'arn:aws:batch:us-west-2:

### Register job definition

In [11]:
# job definition
while True:
    try:
        str_job_definition = f'job-def-{str_image_name}-{int_iteration}'
        dict_response = cls_client.register_job_definition(
            type='container',
            containerProperties={
                'image': f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_image_name}:latest',
                'memory': int_memory_mebibytes,
                'vcpus': int_vcpu,
            },
            jobDefinitionName=str_job_definition,
        )
        # get arn
        str_job_def_arn = dict_response['jobDefinitionArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '199',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 14 Nov 2024 19:16:08 GMT',
                                      'x-amz-apigw-id': 'BQDi4GlUvHcEh-Q=',
                                      'x-amzn-requestid': 'd0f6c5cf-641b-4a44-8a12-1682329f6e03',
                                      'x-amzn-trace-id': 'Root=1-67364c78-2cf866a00b2cb1205fffb30a'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'd0f6c5cf-641b-4a44-8a12-1682329f6e03',
                      'RetryAttempts': 0},
 'jobDefinitionArn': 'arn:aws:batch:us-we

### Submit job

In [14]:
# submit a job (only for testing)
while True:
    try:
        str_job_name = f'job-name-{str_image_name}-{int_iteration}'
        response = cls_client.submit_job(
            jobDefinition=str_job_definition,
            jobQueue=str_job_queue_name,
            jobName=str_job_name,
            arrayProperties={
                'size': int_n_jobs,
            },
        )
        pprint(response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '194',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 14 Nov 2024 19:24:37 GMT',
                                      'x-amz-apigw-id': 'BQEyXFl_vHcEpvA=',
                                      'x-amzn-requestid': '72db6985-c1d2-458a-863c-89f260bf3899',
                                      'x-amzn-trace-id': 'Root=1-67364e75-51a4896a511e4a1326eef2e2'},
                      'HTTPStatusCode': 200,
                      'RequestId': '72db6985-c1d2-458a-863c-89f260bf3899',
                      'RetryAttempts': 0},
 'jobArn': 'arn:aws:batch:us-west-2:83669

### Show arns

In [13]:
print(f'Job Queue ARN: {str_job_queue_arn}')
print(f'Job Definition ARN: {str_job_def_arn}')

Job Queue ARN: arn:aws:batch:us-west-2:836690756591:job-queue/queue-simple-model-test-parse-snowflake-1
Job Definition ARN: arn:aws:batch:us-west-2:836690756591:job-definition/job-def-simple-model-test-parse-snowflake-1:1
